<a href="https://colab.research.google.com/github/andrelison/salesinsight-py2/blob/feat%2Fpipeline-dados/salesinsigth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importar bibliotecas

In [1]:
import json
import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

GERAR DATASET

In [2]:
def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)

df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")

Dataset gerado com 200 registros.


INSPEÇÃO DE DADOS

In [3]:
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

inspecionar_dados(df_bruto)


=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Com

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
...,...,...,...,...,...,...,...,...
195,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
196,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
197,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
198,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


ESQUELETO DA CLASSE

In [13]:
class AnalisadorDeVendas:
    """Organiza o fluxo: carregar, transformar, analisar, visualizar."""

    def __init__(self):
        self.df_bruto = None
        self.df = None
        self.relatorio_limpeza = None
        self.metricas = None
        self.clientes = None

    def carregar(self, caminho):
        """Le o CSV de vendas."""
        self.df_bruto = pd.read_csv(caminho)
        return self.df_bruto

    def transformar(self):
        """Limpa os dados e cria as colunas derivadas (RF03 e RF04)."""
        if self.df_bruto is None:
            raise ValueError("Carregue primeiro o dataset com carregar().")
        df_limpo, self.relatorio_limpeza = tratar_dados(self.df_bruto)
        self.df = criar_colunas_derivadas(df_limpo)
        return self.df

impeza dos dados

In [5]:
def tratar_dados(df):
    """
    Trata o DataFrame de vendas (RF03).
    Retorna: (df_limpo, relatorio), onde relatorio e um dicionario
    com as contagens de registros iniciais, convertidos e finais.
    """
    df = df.copy()
    antes = df.copy()                      # copia para comparar depois
    registros_iniciais = len(df)

    # 1. remover espacos extras nas colunas de texto
    colunas_texto = df.select_dtypes(include="object").columns
    for col in colunas_texto:
        df[col] = df[col].str.strip()
    # True nas linhas em que algum texto mudou depois do strip
    mudou_texto = (
        antes[colunas_texto].fillna("") != df[colunas_texto].fillna("")
    ).any(axis=1)

    # 2. converter data_venda e descartar datas invalidas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    datas_invalidas = int(df["data_venda"].isna().sum())
    df = df[df["data_venda"].notna()].copy()
    datas_convertidas = len(df)

    # 3. ajustar tipos numericos
    tipos_ajustados = 0
    for coluna, tipo in {"quantidade": int, "preco_unitario": float}.items():
        convertida = df[coluna].astype(tipo)
        if convertida.dtype != antes[coluna].dtype:
            tipos_ajustados += 1
        df[coluna] = convertida

    # 4. montar e imprimir o relatorio de limpeza
    # "sem alteracao" = linhas cujo texto e tipos numericos nao mudaram
    if tipos_ajustados == 0:
        sem_alteracao = int((~mudou_texto.loc[df.index]).sum())
    else:
        sem_alteracao = 0

    relatorio = {
        "registros_iniciais": registros_iniciais,
        "espacos_removidos (linhas)": int(mudou_texto.sum()),
        "datas_convertidas (texto -> datetime)": datas_convertidas,
        "datas_invalidas_removidas": datas_invalidas,
        "colunas_numericas_ajustadas": tipos_ajustados,
        "registros_sem_alteracao": sem_alteracao,
        "registros_finais": len(df),
    }

    print("\n=== RELATORIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")

    return df.reset_index(drop=True), relatorio

In [12]:
MESES_PT = {
    1: "Janeiro", 2: "Fevereiro", 3: "Marco", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro",
}

def criar_colunas_derivadas(df):
    """Cria as colunas derivadas exigidas no RF04."""
    df = df.copy()

    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["mes_nome"] = df["mes"].map(MESES_PT)
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    condicoes = [
        df["receita_total"] < 500,
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
        df["receita_total"] >= 5000,
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
    df["faixa_receita_item"] = np.select(condicoes, faixas, default="Nao Classificado")

    return df

In [10]:
def transformar(self):
    """Limpa e cria colunas derivadas (RF03 e RF04)."""
    if self.df_bruto is None:
        raise ValueError("Carregue primeiro o dataset.")
    df_limpo, self.relatorio_limpeza = tratar_dados(self.df_bruto)
    self.df_limpo = criar_colunas_derivadas(df_limpo).reset_index(drop=True)

In [15]:
analisador = AnalisadorDeVendas()
analisador.carregar("vendas.csv")
analisador.transformar()
print(analisador.df.head())
print(analisador.df.dtypes)


=== RELATORIO DE LIMPEZA ===
  registros_iniciais: 200
  espacos_removidos (linhas): 0
  datas_convertidas (texto -> datetime): 194
  datas_invalidas_removidas: 6
  colunas_numericas_ajustadas: 0
  registros_sem_alteracao: 194
  registros_finais: 194
   id_venda data_venda      cliente     produto     categoria    regiao  \
0         1 2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         3 2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
2         4 2025-05-23  Cliente_001     Monitor  Computadores       Sul   
3         5 2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   
4         6 2025-07-03  Cliente_023    Notebook  Computadores   Sudeste   

   quantidade  preco_unitario  receita_total  mes mes_nome trimestre   ano  \
0           2          102.90         205.80    5     Maio        Q2  2025   
1           9         2267.33       20405.97   10  Outubro        Q4  2025   
2           4         1181.72        4726.88    5     Maio     